# Level 2b — From cell types to tissue architecture

### CAJAL Neuromics 2026 · Project 15: Mapping the Spatial Cellular Architecture of the Brain

**~ ½ day · graded**

> *"We know what the cells are. Now: where are they, how are they arranged, and how do they talk?"*

In **2a** you put a trustworthy cell-type label on every MERFISH cell by reference-mapping from the
multiome atlas. This notebook reads *where* those cells sit and *how* neighbouring types communicate —
reproducing the core of **Wang et al. 2025, Figure 2**. The biological arc is the paper's own:

1. **cell types in space** → 2. **niches = the cortical cytoarchitecture** → 3. **neighbourhood
enrichment** (who sits next to whom) → 4. **cell–cell communication** (who signals to whom).

**Learning objectives — by the end you can:**
- Plot annotated cells in physical space and read the developing cortex off the map.
- Define **spatial niches** by clustering each cell's neighbourhood cell-type composition (CLR + k-means)
  and recognise them as the ventricular zone → intermediate zone → cortical layers → white matter.
- Quantify spatial co-localisation with **neighbourhood enrichment** (`squidpy.gr.nhood_enrichment`).
- Infer **cell–cell communication** by cell type (LIANA) and recover the paper's neuregulin /
  somatostatin signalling — and critically compare to the authors' results.

**How to work through this notebook**
- 🔬 **TASK** / 💡 **HINT** / ❓ **QUESTION** / ⚠️ **CHECKPOINT** as before. **Kernel:** `Spatial Brain (SIF)`.
- Yellow **`FutureWarning`** boxes are normal (not errors). Some cells run for **a few minutes**
  (spatial graph, niche clustering, reference mapping) — a slow cell is working, not hung.
- **Data:** reads your own `data/wang2025_merfish/processed/cohort_annotated.h5ad` from 2a and the
  shared read-only RNA reference. A ≥ 32 GB node is recommended.


---
## 0. Setup & load the annotated cohort


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
import squidpy as sq
import anndata as ad

sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, frameon=False, figsize=(4, 4))
%matplotlib inline

DATA = "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C15/data"
REF_PATH = f"{DATA}/wang2025_multiome/processed/wang2025_multiome_rna.h5ad"
GROUND_TRUTH_PATH = f"{DATA}/wang2025_merfish/processed/wang2025_merfish_cells.h5ad"

from spatialbrain import FilePaths

cohort = sc.read_h5ad(FilePaths.dataset("wang2025_merfish").processed / "cohort_annotated.h5ad")
cohort.obs["type_pred"] = cohort.obs["type_pred"].astype("category")
samples = cohort.obs["Sample_ID"].cat.categories.tolist()  # reused throughout for per-sample plots
print(cohort)
print("\ncells per sample:\n", cohort.obs["Sample_ID"].value_counts())

---
## 1. Cell types in space (Fig. 2a)

Plot each section in physical space, coloured by predicted cell type. Even by eye you should see
**layering** — progenitors on one side (the ventricular surface), neurons stacked outward.


🔬 **TASK 1.1 — Plot the tissue.** Draw the first two sections in physical space, coloured by
`type_pred`.

💡 **HINT:** `sq.pl.spatial_scatter` draws cells in physical space; colour by `type_pred` and use
`library_key`/`library_id` to place specific sections side by side (`samples` is from the setup cell).
On hundreds of thousands of cells, turn the polygon shapes off and use a small point size so it renders
fast. See the [squidpy plotting API](https://squidpy.readthedocs.io/).


In [ ]:
# Plot the first two sections (samples[:2]) in space, coloured by `type_pred`.
# HINT: sq.pl.spatial_scatter(cohort, library_key="Sample_ID", library_id=samples[:2], color="type_pred", shape=None, size=2, ncols=2)
...

❓ **QUESTION.** Pick a section: can you already see a spatial gradient of cell types (progenitors vs
mature neurons)? That gradient is the developing cortex's radial organisation — which we formalise next.


---
## 2. The spatial graph — build it and check it

Everything spatial (niches, neighbourhood enrichment) rests on a **spatial neighbour graph**: which
cells are physically close. We build it **per sample** (`library_key`) so there are never edges between
two physically separate sections. Before trusting any downstream result, *look at the graph* — a quick
habit that catches coordinate/units mistakes early.

🔬 **TASK 2.1 — Build the graph and eyeball it.** Build a per-sample 50-nearest-neighbour graph on the
whole cohort (this is what the niche analysis will use). Then, on a **small crop** of one section, draw
the graph to check that edges connect *physically adjacent* cells.

💡 **HINT:** `sq.gr.spatial_neighbors` builds the graph — pass `coord_type="generic"` (these are
arbitrary µm coordinates, not a Visium grid) and a `library_key` so edges never cross sections;
`n_neighs=50` is the neighbourhood the niche step needs. A 50-NN graph is an unreadable hairball to
draw, so for the *picture* subset to one `Sample_ID`, take a coordinate window **a few hundred µm wide**
(big enough to show tissue structure, small enough to stay legible), and rebuild the graph on that crop
with `delaunay=True` (a clean local mesh). `sq.pl.spatial_scatter` with a `connectivity_key` draws the
edges. See the [squidpy Vizgen tutorial](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_vizgen.html).


In [ ]:
# 1. analysis graph: per-sample 50-NN on the whole cohort
# HINT: sq.gr.spatial_neighbors(cohort, coord_type="generic", n_neighs=50, library_key="Sample_ID")
# 2. sanity check: subset to one Sample_ID, take a small coordinate window (crop), build a Delaunay
#    graph on the crop, and draw it with connectivity_key so edges are visible.
...

❓ **QUESTION.** In the crop, is each cell linked to a sensible set of *physically nearby* cells (not to
cells across the section)? If the edges looked random or spanned the whole tissue, something would be
wrong with the coordinates — this is why we look first.


---
## 3. Spatial niches = the cortical cytoarchitecture (Fig. 2a/b)

A **niche** is a recurring local cellular neighbourhood. The authors defined 10 niches by
**k-means on the cell-type identities of each cell's 50 nearest spatial neighbours**, and they turned
out to be the classic cortical domains (VZ/SVZ, intermediate zone, cortical layers 1–6, white matter).

We do the same, with one statistical improvement. Each cell's neighbourhood is a vector of **cell-type
proportions** — *compositional* data that lives on the simplex (it sums to 1). Plain Euclidean k-means on
proportions is not ideal; the principled fix is a **centred-log-ratio (CLR) transform** first, so that
Euclidean distance in CLR space is **Aitchison distance**. (The paper used raw k-means; this is a small,
defensible upgrade.) The neighbourhood composition itself is just `graph @ one-hot(type)` — fast on all
404k cells — so unlike Leiden/GMM niche callers this scales trivially and we cluster **all samples
together** into shared niches.

🔬 **TASK 3.1 — Call the niches.** Build the neighbour cell-type composition, CLR-transform it, and
k-means it into 10 niches (`cohort.obs["niche"]`).

💡 **HINT:** The composition is a matrix product — the adjacency graph
(`cohort.obsp["spatial_connectivities"]`) times a one-hot encoding of `type_pred` (`pd.get_dummies`)
gives, per cell, how many of each type sit among its neighbours; row-normalise to proportions. For the
CLR, take `log` (add a small pseudocount — most proportions are zero) and subtract each row's mean.
Then cluster with `sklearn.cluster.KMeans(n_clusters=10)`; fix `random_state` for reproducibility.


In [ ]:
# 1. neighbour cell-type composition (graph @ one-hot(type), row-normalised to proportions)
# 2. CLR transform (add a small pseudocount, take log, subtract the per-cell mean-of-logs)
# 3. KMeans(n_clusters=10) on the CLR matrix -> cohort.obs["niche"]
...
cohort.obs["niche"] = ...

⚠️ **CHECKPOINT.** You get exactly **10 niches** (fixed k). Now visualise them in space and read each
niche's identity from its cell-type composition.


In [ ]:
# niches in space (Fig. 2a, right)
sq.pl.spatial_scatter(
    cohort, library_key="Sample_ID", library_id=samples[:2], color="niche", shape=None, size=2, ncols=2, figsize=(8, 8)
)

# composition: proportion of each cell type within each niche (Fig. 2b)
comp = pd.crosstab(cohort.obs["niche"], cohort.obs["type_pred"], normalize="index")
import seaborn as sns

plt.figure(figsize=(14, 5))
sns.heatmap(comp, cmap="viridis")
plt.title("cell-type composition per niche")
plt.show()

🔬 **TASK 3.2 — Name the niches biologically.** From the composition heatmap and the spatial plots,
assign each of the 10 niches a cortical-domain label. The authors' domains were: VZ/SVZ, intermediate
zone, cortical layers 1–6/subplate, white matter & meninge, and a dorsal-LGE germinal zone. Which niche
is progenitor-rich (the **VZ/SVZ**)? Which are single-layer excitatory (the **cortical layers**)? Which
is oligodendrocyte-rich (the **white matter**)?

💡 **HINT:** progenitors = RG-*/IPC-*; deep layers = EN-L5/L6; upper layers = EN-L2_3/L4; white matter =
Oligodendrocyte-rich. Note `IN-dLGE-Immature` appears **both** in its own dorsal-LGE niche **and**
dispersed in white matter (they're the future white-matter interstitial interneurons) — so don't fold
the dLGE germinal zone into "white matter"; it's a distinct 10th domain.


🚀 **Going further (optional — open-ended).** Are these niches a robust finding or an artefact of the
parameters? Re-call them for a few combinations of neighbourhood size (`n_neighs`) and cluster count
(`k`), and measure how much the partitions agree (`sklearn.metrics.adjusted_rand_score`). Which domains
(VZ, white matter, a cortical layer) re-emerge every time, and which are fragile? Defend a choice of
parameters biologically, not just by a score.


---
## 4. Neighbourhood enrichment — who sits next to whom (Fig. 2c)

Niches show *domains*; neighbourhood enrichment tests, pairwise, which cell **types** are spatial
neighbours more (or less) than chance. The paper's Fig. 2c is the **PFC section at infancy**, run
**per sample** — so we pick that section (the oldest PFC sample) and build a *tighter* graph for it: a
few immediate neighbours (`n_neighs=6`) rather than the 50 used for niche calling, because enrichment
asks about *direct* adjacency, not broad neighbourhood composition. Beyond each excitatory type
clustering with itself (layer specificity), look for **excitatory↔inhibitory** pairs — the paper
highlighted **EN-L4-IT ↔ IN-MGE-SST** and **EN-IT-Immature ↔ IN-CGE-VIP**.

🔬 **TASK 4.1 — Neighbourhood enrichment on the PFC-infancy section.** Subset to the oldest PFC sample,
build a *tight* spatial graph, run neighbourhood enrichment on `type_pred`, and plot it.

💡 **HINT:** Rebuild `sq.gr.spatial_neighbors` on just that subset with a small `n_neighs` (direct
adjacency, not the broad 50), then `sq.gr.nhood_enrichment(cluster_key="type_pred")`, then
`sq.pl.nhood_enrichment`. Set a `seed` so the permutation z-scores are reproducible, and a diverging
colour scale centred near 0 reads best. ([squidpy docs](https://squidpy.readthedocs.io/).)


In [ ]:
# Subset to the oldest ("infancy") PFC sample, build a 6-NN spatial graph, run nhood_enrichment on
# `type_pred`, and plot it (cmap="inferno", vmin=-50, vmax=100).
...

❓ **QUESTION.** Do the excitatory-neuron types form their own neighbourhoods (positive self-enrichment)?
Can you spot a positive EN↔IN enrichment? Spatial adjacency is a *prerequisite* for the short-range
signalling we look at next.


---
## 5. (Light) the reference atlas (Fig. 1b)

A quick look at the multiome reference these labels came from — the molecular taxonomy, on its own UMAP.


In [ ]:
ref = sc.read_h5ad(REF_PATH)
if "X_wnn.umap" in ref.obsm:
    ref.obsm["X_umap"] = ref.obsm["X_wnn.umap"]
    sc.pl.umap(ref, color="type", legend_loc="on data", legend_fontsize=5, size=2)

---
## 6. Cell–cell communication (Fig. 2e/f)

**A subtlety worth understanding:** the 300-gene MERFISH panel is a *cell-typing* panel, so it contains
almost no complete **ligand–receptor pairs**. For the pathways of interest the *ligand* `SST` is on the
panel but its receptors `SSTR1/2` are not, and none of the `NRG1/2/3`–`ERBB3/4` pairs are — so you
**cannot score LR interactions from the measured spatial data alone**. The authors ran CellChat on the
**snRNA** side instead. We do the equivalent, keeping everything on our spatial cells: **reference
mapping imputes the transcriptome** (including the LR genes) onto the spatial cells — the same mapping
that gave us labels in 2a, and whose imputation we validated with held-out genes — then we run **LIANA**
by cell type. *(You can confirm the gap yourself: `"SST" in cohort.var_names` is True but
`"SSTR2" in cohort.var_names` is False.)*

For speed we subsample query + reference (communication aggregates by type) and impute only the
ligand–receptor genes; `only_yx=True` keeps CellMapper's kNN to the one direction we need.


In [ ]:
import liana as li
from cellmapper import CellMapper


def strat(adata, col, n, seed=0):
    if n >= adata.n_obs:
        return adata.copy()
    rng = np.random.default_rng(seed)
    idx = []
    for _, s in pd.Series(range(adata.n_obs)).groupby(adata.obs[col].values, observed=True):
        idx.extend(rng.choice(s.values, size=min(len(s), max(1, round(len(s) * n / adata.n_obs))), replace=False))
    return adata[np.sort(np.array(idx))].copy()


# LR-resource genes present in the reference; impute panel + LR onto a spatial subsample
lr = set()
for c in ("ligand", "receptor"):
    for g in li.resource.select_resource("consensus")[c]:
        lr.update(str(g).split("_"))
ref.obs["type"] = ref.obs["type"].astype("category")
genes = sorted(set(cohort.var_names) | (lr & set(ref.var_names)))
q = strat(cohort, "type_pred", 40000)
r = strat(ref[:, genes].copy(), "type", 50000)

cm = CellMapper(q, r)
cm.compute_fast_cca(n_comps=50)
libsize = np.asarray(q.layers["counts"].sum(1)).ravel()
cm.map(layer_key="X", use_rep="X_cca", n_neighbors=30, knn_method="pynndescent", target_libsize=libsize, only_yx=True)

lr_genes = sorted(lr & set(genes))
imp = ad.AnnData(X=cm.query_imputed[:, lr_genes].X, obs=q.obs[["type_pred"]].copy(), var=pd.DataFrame(index=lr_genes))
li.mt.natmi(imp, groupby="type_pred", expr_prop=0.1, use_raw=False, verbose=False)
res = imp.uns["liana_res"]
print(res.shape, "interactions")

🔬 **TASK 5.1 — Recover the paper's signalling.** From `res`, pull the interactions where the
**sender** (`source`) is `IN-MGE-SST` and the ligand is `SST`/`CORT` (somatostatin), and where the
sender is `EN-IT-Immature` and the ligand is `NRG1`/`NRG2`/`NRG3` (neuregulin). Sort by `expr_prod`.
Are the top targets excitatory neurons (for SST) and interneurons (for NRG)?

💡 **HINT:** `res` is a dataframe of interactions (one row per source→target ligand–receptor pair).
Filter it by `source` and `ligand_complex`, then sort by `expr_prod` (descending) to see the strongest
targets. See the [LIANA docs](https://liana-py.readthedocs.io/) for the output columns.


In [ ]:
# For each pathway, filter `res` by sender + ligand and show the top targets.
...

⚠️ **CHECKPOINT.** **Somatostatin** from `IN-MGE-SST` should target excitatory neurons (EN-L2_3/L4/L5/L6-IT,
EN-IT-Immature) — the paper's Fig. 2f. **Neuregulin** from `EN-IT-Immature` should target interneurons and
glia — Fig. 2e. Note the *receptors* LIANA reports (`GRM7`, `EGFR`, `NETO2`) differ from the paper's
CellChatDB pairs (`SSTR`, `ERBB`) — a **resource difference**, not a biological one.

❓ **QUESTION.** The sender→receiver cell-type story matches the paper, but the receptor annotations
differ. Why might two ligand–receptor databases (LIANA-consensus vs CellChatDB) disagree on receptors,
and how would you decide which to trust for a specific claim?


---
## 7. The biological story — did we rebuild the cortex? (compare to the authors)

Finally, load the authors' original annotations (now allowed) and compare — but the question isn't a
number, it's biology: **did our niches recover the same cortical architecture, and our communication the
same signalling?** The authors made their labels/niches *independently* (clustering + markers +
k-means), so agreement is genuine cross-validation.


In [ ]:
truth = sc.read_h5ad(GROUND_TRUTH_PATH, backed="r")
cohort.obs["niche_true"] = pd.Categorical(truth.obs.loc[cohort.obs_names, "niche_name"].values)
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(cohort.obs["niche_true"].astype(str), cohort.obs["niche"].astype(str))
nmi = normalized_mutual_info_score(cohort.obs["niche_true"].astype(str), cohort.obs["niche"].astype(str))
print(f"niche agreement vs authors: ARI={ari:.3f}  NMI={nmi:.3f}")

# which of our niches maps to which named cortical domain?
xt = pd.crosstab(cohort.obs["niche"], cohort.obs["niche_true"], normalize="index")
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.heatmap(xt, cmap="magma")
plt.title("our niches (rows) vs authors' named domains (cols)")
plt.show()

🔬 **TASK 6.1 — Write the story (3–5 sentences).** Using the crosstab, state which cortical domains you
recovered and how cleanly they map to the authors'. Then connect it back to development: progenitors in
the VZ/SVZ, excitatory neurons laminated inside-out, interneurons and oligodendrocytes in the white
matter — and the reciprocal EN↔IN (neuregulin / somatostatin) signalling that helps build it. Where did
your reconstruction diverge from the paper, and is that a real biological difference or a
method/database artefact?


🚀 **Going further (optional — open-ended).** We imputed the transcriptome to recover the off-panel
receptors. What can you claim from the **measured** 300 genes alone? Re-run LIANA using only the
ligand–receptor pairs whose *both* partners are on the panel, and compare the top senders/receivers to
the imputed result. Which of the paper's findings survive without imputation, and which depend on it?


---
## Where this is heading

You've reproduced the heart of Wang et al.'s Figure 2 — cell types in space, the niche cytoarchitecture,
neighbourhood enrichment, and cell–cell communication — and critically compared it to the authors'
results. **Level 3** takes the brakes off: you'll bring in the multiome **ATAC** side and ask a question
the paper left open (e.g. the fate of white-matter interneurons, or the Tri-IPC gliogenesis programme).


In [ ]:
import session_info2

session_info2.session_info(os=True, cpu=True)